# Enterprise Retail Sales & Customer Analytics Platform

## Feature Engineering

### Objective
Create new business features and KPIs that improve analysis and support advanced SQL and Power BI dashboards.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("clean_superstore.csv")

df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,State/Province,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Days
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512,4
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870,4
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717,4
3,4,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748,4
4,5,US-2023-141817,2023-01-05,2023-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840,7


- <b>Feature 1 – Shipping Days</b>

In [3]:
df["Shipping Days"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days

df[["Order Date","Ship Date","Shipping Days"]].head()

,Order Date,Ship Date,Shipping Days
0,2023-01-03,2023-01-07,4
1,2023-01-04,2023-01-08,4
2,2023-01-04,2023-01-08,4
3,2023-01-04,2023-01-08,4
4,2023-01-05,2023-01-12,7


- <b>Feature 2 – Profit Margin %</b>

In [4]:
df["Profit Margin %"] = np.where(
    df["Sales"] != 0,
    (df["Profit"] / df["Sales"]) * 100,
    0
)

df[["Sales","Profit","Profit Margin %"]].head()

,Sales,Profit,Profit Margin %
0,16.448,5.5512,33.75
1,3.540,-5.4870,-155.00
2,11.784,4.2717,36.25
3,272.736,-64.7748,-23.75
4,19.536,4.8840,25.00


- <b>Feature 3 – Order Year</b>

In [5]:
df["Order Year"] = df["Order Date"].dt.year

df["Order Year"].value_counts()

Order Year
2026    3379
2025    2634
2024    2130
2023    2051
Name: count, dtype: int64

- <b>Feature 4 – Order Month</b>

In [6]:
df["Order Month"] = df["Order Date"].dt.month_name()

df["Order Month"].head()

0    January
1    January
2    January
3    January
4    January
Name: Order Month, dtype: object

- <b>Feature 5 – Order Quarter</b>

In [7]:
df["Order Quarter"] = "Q" + df["Order Date"].dt.quarter.astype(str)

df["Order Quarter"].head()

0    Q1
1    Q1
2    Q1
3    Q1
4    Q1
Name: Order Quarter, dtype: object

- <b>Feature 6 – Loss Orders</b>

In [8]:
df["Loss Order"] = np.where(
    df["Profit"] < 0,
    "Yes",
    "No"
)

df["Loss Order"].value_counts()

Loss Order
No     8293
Yes    1901
Name: count, dtype: int64

- <b>Feature 7 – High Value Orders</b>

In [9]:
high_value = df["Sales"].quantile(0.75)

df["High Value Order"] = np.where(
    df["Sales"] >= high_value,
    "Yes",
    "No"
)

df["High Value Order"].value_counts()

High Value Order
No     7644
Yes    2550
Name: count, dtype: int64

- <b>Feature 8 – Order Size</b>

In [10]:
conditions = [
    df["Quantity"] <= 2,
    df["Quantity"].between(3, 5),
    df["Quantity"] > 5
]

choices = [
    "Small",
    "Medium",
    "Large"
]

df["Order Size"] = np.select(
    conditions,
    choices,
    default="Small"
)

df["Order Size"].value_counts()

Order Size
Medium    4916
Small     3373
Large     1905
Name: count, dtype: int64

- <b>Feature 9 – Customer Lifetime Sales</b>

In [11]:
customer_sales = (
    df.groupby("Customer ID")["Sales"]
      .sum()
      .reset_index()
)

customer_sales.columns = [
    "Customer ID",
    "Lifetime Sales"
]

df = df.merge(
    customer_sales,
    on="Customer ID",
    how="left"
)

df[["Customer ID","Lifetime Sales"]].head()

,Customer ID,Lifetime Sales
0,DP-13000,1050.636
1,PO-19195,1056.858
2,PO-19195,1056.858
3,PO-19195,1056.858
4,MB-18085,1428.231


- <b>Feature 10 – Customer Lifetime Profit</b>

In [12]:
customer_profit = (
    df.groupby("Customer ID")["Profit"]
      .sum()
      .reset_index()
)

customer_profit.columns = [
    "Customer ID",
    "Lifetime Profit"
]

df = df.merge(
    customer_profit,
    on="Customer ID",
    how="left"
)

df[["Customer ID","Lifetime Profit"]].head()

,Customer ID,Lifetime Profit
0,DP-13000,241.4524
1,PO-19195,-49.7009
2,PO-19195,-49.7009
3,PO-19195,-49.7009
4,MB-18085,117.8057


- <b>Feature 11 – Top Customer Flag</b>

In [13]:
threshold = df["Lifetime Sales"].quantile(0.90)

df["Top Customer"] = np.where(
    df["Lifetime Sales"] >= threshold,
    "Yes",
    "No"
)

df["Top Customer"].value_counts()

Top Customer
No     9163
Yes    1031
Name: count, dtype: int64

- <b>Feature 12 – Save Final Dataset</b>

In [14]:
df.to_csv("superstore_feature_engineered.csv", index=False)

print("Feature Engineered Dataset Saved Successfully!")

Feature Engineered Dataset Saved Successfully!
